# Ajuste Fino do YOLO

No material anterior, montamos e treinamos do zero um detector no estilo do YOLO. Na prática, ninguém treina um detector do zero: parte-se de um modelo pré-treinado em um dataset grande e faz-se o **ajuste fino** (*fine-tuning*) para as classes do problema, como fizemos com classificadores na transferência de aprendizado.

Vamos usar a biblioteca **Ultralytics** (`pip install -U ultralytics`), que implementa as versões recentes do YOLO. O YOLO26, a versão que usamos aqui, precisa da Ultralytics 8.4 ou mais nova. O caminho é:

1. fazer inferência com um modelo pré-treinado;
2. entender o formato de dataset que o YOLO espera e ajustar o modelo em um dataset pronto;
3. reunir as imagens e rotular o **seu próprio dataset** de detecção e ajustar o YOLO para ele.

In [ ]:
import os
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageOps
from ultralytics import YOLO
from ultralytics.utils import ASSETS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Inferência

A Ultralytics distribui o **YOLO26**, a versão mais recente do YOLO, em cinco tamanhos: `n` (*nano*), `s`, `m`, `l` e `x`. Quanto maior, mais preciso e mais lento. Todos foram treinados no **COCO**, um dataset com 80 classes de objetos do dia a dia: pessoas, carros, animais, móveis, comida. Os pesos são baixados na primeira vez que o modelo é carregado.

In [ ]:
model = YOLO("yolo26n.pt")

print(f"{len(model.names)} classes")
print(list(model.names.values())[:20])

O método `predict` recebe um caminho, uma URL, uma imagem PIL ou uma lista deles, e devolve uma lista de resultados, um por imagem. Vamos usar uma foto de exemplo que vem com a biblioteca. O `conf` é o limiar de confiança. O YOLO26 dispensa a supressão de não máximos do material anterior: ele é treinado para prever uma única caixa por objeto, então a saída da rede já é a lista final de detecções. Nas versões anteriores, como o YOLO11, a supressão vem embutida no `predict`, com o limiar de IoU no argumento `iou`.

O `plot` de um resultado devolve a imagem com as caixas desenhadas, com os canais na ordem BGR, então invertemos a ordem para o matplotlib.

In [ ]:
results = model.predict(ASSETS / "bus.jpg", conf=0.25)
result = results[0]

plt.figure(figsize=(6, 8))
plt.imshow(result.plot()[:, :, ::-1])   # BGR -> RGB
plt.axis("off")
plt.show()

As caixas ficam em `result.boxes`. Cada uma tem a classe (`cls`), a confiança (`conf`) e as coordenadas em vários formatos: `xyxy` são os cantos em pixels e `xywhn` é o formato do material anterior, com centro e tamanho divididos pelo tamanho da imagem.

In [ ]:
for box in result.boxes:
    label = model.names[int(box.cls)]
    x0, y0, x1, y1 = box.xyxy[0].tolist()
    print(f"{label:10s} confiança {box.conf.item():.2f}, cantos ({x0:.0f}, {y0:.0f}) e ({x1:.0f}, {y1:.0f})")

## Ajuste fino

O modelo pré-treinado só conhece as 80 classes do COCO. Para detectar outras coisas, fazemos o ajuste fino: a rede mantém as camadas que extraem características, que servem para qualquer imagem, e a camada final é trocada por uma com o número de classes do novo problema. A Ultralytics faz essa troca sozinha, a partir da lista de classes do dataset. Só precisamos entregar os dados no formato certo.

### O formato do dataset

O YOLO espera as imagens e as anotações em pastas separadas, com uma subpasta para cada conjunto:

```
dataset/
    data.yaml
    images/train/  images/val/
    labels/train/  labels/val/
```

Cada imagem `images/train/foto.jpg` tem um arquivo de anotação `labels/train/foto.txt`, com uma linha por objeto:

```
classe x y w h
```

no mesmo formato do material anterior: o índice da classe, o centro e o tamanho da caixa, divididos pelo tamanho da imagem. Uma imagem sem objetos pode ficar sem arquivo de anotação; ela serve de exemplo de fundo.

Pode haver também uma pasta `test`, com imagens que só são usadas no fim, para a avaliação final.

O `data.yaml` diz onde estão as pastas e o nome de cada classe, na ordem dos índices.

In [ ]:
def write_data_yaml(dataset_dir, classes):
    path = os.path.join(dataset_dir, "data.yaml")
    with open(path, "w") as f:
        f.write(f"path: {os.path.abspath(dataset_dir)}\n")
        f.write("train: images/train\n")
        f.write("val: images/val\n")
        if os.path.exists(os.path.join(dataset_dir, "images", "test")):
            f.write("test: images/test\n")
        f.write("names:\n")
        for index, name in enumerate(classes):
            f.write(f"  {index}: {name}\n")
    return path

### Exemplo: equipamentos de proteção individual

Um dos usos mais comuns de detecção na indústria é a segurança do trabalho: câmeras em obras e fábricas verificam se cada pessoa está usando os **equipamentos de proteção individual** (EPIs) exigidos, como capacete, colete, luvas, botas e óculos. Vamos usar o **construction-ppe**, um dataset de fotos de obras publicado pela Ultralytics no GitHub, com 1132 imagens de treino, 143 de validação e 141 de teste.

São 11 classes. Além de pessoas (`Person`) e de cada equipamento (`helmet`, `vest`, `gloves`, `boots`, `goggles`), há classes para a **ausência** de cada um: `no_helmet` marca uma cabeça sem capacete, `no_gloves` uma mão sem luva, e assim por diante. Na prática, o que interessa é justamente detectar quem está sem o equipamento.

O dataset tem os problemas típicos de dados reais, e vale olhar para eles antes de treinar:

- **nomes inconsistentes**: a classe 5 se chama `none`, mas, olhando as caixas, ela marca torsos sem colete. Na nossa lista de classes ela vira `no_vest`. Os índices dos arquivos de anotação continuam os mesmos; só o nome muda;
- **classes desbalanceadas**: há quase 1800 pessoas no treino, mas menos de 100 caixas de `no_boots`;
- **objetos pequenos**: luvas e óculos ocupam menos de 1% da área da imagem;
- **arquivos sobrando**: algumas anotações do treino não têm imagem correspondente. A Ultralytics as ignora.

In [ ]:
ppe_dir = "data/construction-ppe"
url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
ppe_classes = ["helmet", "gloves", "vest", "boots", "goggles", "no_vest", "Person",
               "no_helmet", "no_goggle", "no_gloves", "no_boots"]

if not os.path.exists(ppe_dir):
    urllib.request.urlretrieve(url, "data/construction-ppe.zip")
    with zipfile.ZipFile("data/construction-ppe.zip") as z:
        z.extractall(ppe_dir)

ppe_yaml = write_data_yaml(ppe_dir, ppe_classes)
print(open(ppe_yaml).read())

train_labels_dir = os.path.join(ppe_dir, "labels", "train")
n_boxes = [0] * len(ppe_classes)
for label_name in os.listdir(train_labels_dir):
    for line in open(os.path.join(train_labels_dir, label_name)).read().splitlines():
        n_boxes[int(line.split()[0])] += 1

for name, count in zip(ppe_classes, n_boxes):
    print(f"{name:10s} {count:5d} caixas no treino")

A função `show_labels` lê um arquivo de anotação e desenha as caixas sobre a imagem, com uma cor para cada classe. Ela serve para conferir se as anotações estão certas antes de treinar, e vamos usá-la de novo no seu dataset.

In [ ]:
colors = [plt.cm.tab20(index) for index in range(20)]


def show_labels(ax, image_path, label_path, classes):
    image = Image.open(image_path).convert("RGB")
    W, H = image.size
    ax.imshow(image)
    ax.axis("off")

    if not os.path.exists(label_path):
        return   # imagem de fundo, sem objetos
    for line in open(label_path).read().splitlines():
        label, x, y, w, h = line.split()
        color = colors[int(label) % len(colors)]
        x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
        ax.add_patch(plt.Rectangle((x - w / 2, y - h / 2), w, h, linewidth=2, edgecolor=color, facecolor="none"))
        ax.text(x - w / 2, y - h / 2, classes[int(label)], color="black", backgroundcolor=color, fontsize=7)


train_images_dir = os.path.join(ppe_dir, "images", "train")
names = random.Random(0).sample(sorted(os.listdir(train_images_dir)), 4)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, name in zip(axes, names):
    label_name = os.path.splitext(name)[0] + ".txt"
    show_labels(ax, os.path.join(train_images_dir, name), os.path.join(train_labels_dir, label_name), ppe_classes)
plt.tight_layout()
plt.show()

### Treinando

O `train` recebe o `data.yaml` e os hiperparâmetros principais:

- `epochs`: o número de épocas;
- `imgsz`: o tamanho para o qual as imagens são redimensionadas, múltiplo de 32. O padrão é 640;
- `batch`: o tamanho do lote;
- `project` e `name`: a pasta onde ficam os resultados, aqui `runs/ppe`.

O `project` vai com o caminho absoluto: as versões recentes da Ultralytics colocam um caminho relativo dentro de outra pasta, `runs/detect/`, e os pesos acabariam em `runs/detect/runs/ppe`. Com o caminho absoluto, eles ficam onde pedimos.

O otimizador, a taxa de aprendizado e o aumento de dados vêm configurados. Ao fim de cada época, o modelo é avaliado no conjunto de validação, e os pesos da melhor época ficam em `runs/ppe/weights/best.pt`.

In [ ]:
runs_dir = os.path.abspath("runs")

model = YOLO("yolo26n.pt")
model.train(data=ppe_yaml, epochs=30, imgsz=640, batch=16, project=runs_dir, name="ppe", exist_ok=True)

No início do registro do treino, duas linhas mostram o ajuste fino acontecendo: `Overriding model.yaml nc=80 with nc=11` troca as 80 classes do COCO pelas 11 classes do dataset, e `Transferred ... items from pretrained weights` diz quantos tensores de pesos vieram do modelo pré-treinado. A classe `Person` também existe no COCO, como `person`, e a linha `Remapped 1/11 cls head rows` mostra que a Ultralytics reaproveitou os pesos dela na nova cabeça; as outras 10 classes começam do zero. Os que ficam de fora são os da parte da cabeça que prevê as classes, cujo formato depende do número de classes; esses começam do zero.

### Avaliação

A métrica padrão de detecção é o **mAP** (*mean average precision*). Para cada classe, as detecções são ordenadas pela confiança e, conforme o limiar de confiança desce, a precisão e a revocação vão mudando; a **AP** é a área sob essa curva de precisão por revocação. O mAP é a média da AP sobre as classes:

$$
\text{mAP} = \frac{1}{C} \sum_{c=1}^{C} \text{AP}_c
$$

em que $C$ é o número de classes. Uma detecção só conta como acerto se tiver IoU acima de um limiar com uma caixa verdadeira. O **mAP50** usa o limiar 0,5. O **mAP50-95** é a média para os limiares de 0,5 a 0,95, e cobra caixas mais justas.

Com várias classes, vale olhar também a AP de cada uma: um mAP razoável pode esconder uma classe que o modelo quase nunca acerta. E num sistema de segurança, a classe que importa é a de quem está sem o equipamento, como `no_helmet`, não a média.

Avaliamos no conjunto de validação, que foi usado durante o treino para escolher a melhor época, e no de teste, que o modelo nunca viu. O número a reportar é o do teste.

In [ ]:
ppe_model = YOLO(os.path.join(runs_dir, "ppe", "weights", "best.pt"))
val_metrics = ppe_model.val(data=ppe_yaml, split="val", verbose=False)
test_metrics = ppe_model.val(data=ppe_yaml, split="test", verbose=False)

print(f"validação: mAP50 {val_metrics.box.map50:.3f}, mAP50-95 {val_metrics.box.map:.3f}")
print(f"teste:     mAP50 {test_metrics.box.map50:.3f}, mAP50-95 {test_metrics.box.map:.3f}")

print("\nAP50 no teste, da melhor para a pior classe:")
test_ap50 = sorted(zip(test_metrics.box.ap50, test_metrics.box.ap_class_index), reverse=True)
for ap50, index in test_ap50:
    print(f"{ppe_classes[index]:10s} {ap50:.3f}")

In [ ]:
test_dir = os.path.join(ppe_dir, "images", "test")
paths = [os.path.join(test_dir, name) for name in sorted(os.listdir(test_dir))[:8]]
results = ppe_model.predict(paths, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, result in zip(axes.flat, results):
    ax.imshow(result.plot()[:, :, ::-1])
    ax.axis("off")
plt.tight_layout()
plt.show()

Compare a AP de cada classe com o que vimos nos dados. Os equipamentos, mesmo os pequenos como luvas e óculos, costumam ser bem detectados. As classes de **ausência** ficam bem atrás, e `no_boots`, a mais rara, quase nunca é encontrada. Detectar a falta de algo é mais difícil: uma cabeça sem capacete se parece com qualquer cabeça, e o modelo precisa aprender que a diferença está no que não aparece. Além disso, essas classes têm bem menos exemplos.

O problema é que, num sistema de segurança, as classes de ausência são exatamente as que importam: um alarme de `no_helmet` que quase nunca dispara não serve para nada, por melhor que seja o mAP médio. Num projeto real, os próximos passos seriam atacar essas classes: juntar e rotular mais imagens de pessoas sem equipamento, usar um modelo maior e um `imgsz` maior, ou mudar a formulação do problema, por exemplo detectando só pessoas e cabeças e classificando depois se cada uma está com o equipamento.

## O seu detector

Agora cada um monta o seu dataset de detecção, do começo ao fim: reunir as imagens, rotular os objetos, organizar os arquivos no formato do YOLO e fazer o ajuste fino. O código é por sua conta; as funções `write_data_yaml` e `show_labels` e as células do exemplo acima podem ser reaproveitadas.

### Exercício: as fotos

Escolha de 1 a 3 classes de objetos e reúna as imagens. Elas podem vir de onde for mais prático:

- **fotos suas**, tiradas com o celular;
- **imagens da internet**, baixadas à mão ou por *scraping*. A biblioteca `icrawler` (`pip install icrawler`) baixa os resultados de uma busca de imagens: `BingImageCrawler(storage={"root_dir": "data/meu_detector/fotos/caneca"}).crawl(keyword="caneca", max_num=100)`. Ela numera os arquivos a partir de `000001` em cada pasta, então use uma pasta por busca;
- **quadros de um vídeo**, extraídos com o `cv2.VideoCapture`, pegando um quadro a cada um ou dois segundos;
- uma mistura dessas fontes, que costuma dar a maior variedade.

Algumas dicas:

- objetos com contorno bem definido são mais fáceis de rotular e de detectar: canecas, canetas, chaves, frutas, tênis, controles remotos;
- varie o fundo, a iluminação, a distância e o ângulo, e tenha imagens com vários objetos, inclusive de classes diferentes. Um modelo treinado só com fotos na mesma mesa aprende a mesa; um treinado só com fotos de catálogo, com o objeto centralizado em fundo branco, não reconhece o objeto em uma cena real;
- algumas imagens sem nenhum dos objetos ajudam o modelo a não detectar coisas no fundo;
- comece com cerca de 50 a 100 imagens: cada uma vai ser rotulada à mão.

Imagens da internet e de vídeo têm cuidados próprios. A busca devolve resultados errados, ilustrações e imagens repetidas; olhe todas e descarte o que não serve. Imagens repetidas ou quadros quase iguais de um vídeo que caiam uma no treino e outra na validação deixam a validação otimista demais. E imagens da internet têm direitos autorais: para um trabalho de aula, tudo bem, mas não publique o dataset.

Coloque as imagens em uma pasta, por exemplo `data/meu_detector/fotos`, e prepare-as para a rotulação, salvando o resultado em outra pasta, como `data/meu_detector/images`:

- fotos de celular às vezes vêm "deitadas", com a rotação guardada só nos metadados (EXIF). O `ImageOps.exif_transpose` do PIL aplica essa rotação à imagem;
- imagens grandes deixam a rotulação e o treino lentos. Reduza o maior lado para cerca de 1024 pixels, mantendo a proporção (`Image.thumbnail`);
- salve tudo em JPEG, convertendo para RGB (`image.convert("RGB")`), já que imagens da internet podem vir em PNG com transparência, WebP ou GIF. Se o seu celular salva em HEIC, configure a câmera para JPEG ou converta antes.

Mostre algumas das imagens preparadas para conferir se estão na orientação certa.

In [ ]:
# 1. percorra as imagens em data/meu_detector/fotos, inclusive as subpastas
# 2. aplique a rotação do EXIF, converta para RGB e reduza o maior lado para ~1024 pixels
# 3. salve em JPEG em data/meu_detector/images, com nomes que não se repitam entre as subpastas
# 4. mostre algumas imagens e o total de imagens preparadas

### Exercício: rotulando

Rotular é desenhar à mão uma caixa em volta de cada objeto, em cada imagem, e escolher a classe. Todas as ferramentas abaixo exportam no formato YOLO. Exceto o Roboflow, todas são de código aberto:

| ferramenta | como usar | observação |
|---|---|---|
| [Roboflow](https://roboflow.com) | no site, com conta | a mais usada em tutoriais de YOLO, com rotulação assistida e projetos compartilhados. Não é de código aberto e, no plano gratuito, **os projetos ficam públicos**: qualquer pessoa pode ver e baixar as suas fotos. Não use fotos de pessoas ou da sua casa que você não queira publicar |
| [makesense.ai](https://www.makesense.ai) | no navegador, sem instalar nem criar conta | a mais simples; as imagens não saem do computador. Não salva o projeto: rotule tudo em uma sessão ou exporte no meio do caminho |
| [Label Studio](https://labelstud.io) | `pip install label-studio` e `label-studio start` | roda localmente no navegador e salva o projeto |
| [X-AnyLabeling](https://github.com/CVHub520/X-AnyLabeling) | programa de computador | sugere caixas automaticamente com modelos como o SAM e o próprio YOLO; você só corrige |
| [CVAT](https://www.cvat.ai) | instalado com Docker ou no site | a mais completa, feita para equipes rotulando juntas |

Qualquer que seja a ferramenta:

1. crie as classes e anote a ordem delas: no formato YOLO, a classe é um índice, e a sua lista de classes no notebook precisa estar na mesma ordem. A exportação traz um arquivo com os nomes na ordem certa (`classes.txt` ou `data.yaml`); confira por ele, porque algumas ferramentas, como o Roboflow, ordenam as classes em ordem alfabética;
2. desenhe caixas justas em volta dos objetos. Rotule **todos** os objetos das classes escolhidas: um objeto sem caixa ensina o modelo que aquilo é fundo;
3. exporte no formato YOLO e extraia o `.zip` em uma pasta, por exemplo `data/meu_detector/export`.

No makesense: *Get Started*, arraste as imagens de `data/meu_detector/images`, escolha *Object Detection*, crie as classes, rotule e exporte em *Actions → Export Annotations → YOLO*.

No Roboflow, ao gerar a versão do dataset, remova o pré-processamento *Resize* e não aplique aumento de dados: o YOLO já faz aumento de dados no treino, e cópias aumentadas de uma mesma foto acabariam em treino e validação ao mesmo tempo. Exporte no formato YOLO mais recente da lista (*YOLO26* ou *YOLOv11*): o formato dos arquivos é o mesmo.

Cada ferramenta organiza a exportação de um jeito: o makesense entrega só os `.txt`; o Label Studio, pastas `images` e `labels`; o Roboflow, pastas `train`, `valid` e `test`. Junte tudo em duas pastas, uma de imagens e uma de anotações, em que cada imagem `foto.jpg` tenha a anotação `foto.txt`. Cuidado: algumas ferramentas renomeiam os arquivos na exportação; nesse caso, use as imagens que vieram na exportação, e não as originais.

In [ ]:
# 1. junte as imagens e os .txt da exportação em data/meu_detector/images e data/meu_detector/labels
# 2. defina a lista de classes, na mesma ordem dos índices da exportação

classes = [
    # "nome_da_classe_0", "nome_da_classe_1", ...
]

### Exercício: conferindo as anotações

Antes de treinar, confira as anotações. Erros aqui não dão mensagem de erro no treino: o modelo simplesmente aprende errado.

- todo `.txt` corresponde a uma imagem com o mesmo nome?
- cada linha tem 5 valores, e o índice da classe existe na sua lista?
- quantas imagens foram rotuladas, e quantas caixas há de cada classe? Classes com bem menos caixas que as outras vão ser mais difíceis, como vimos no exemplo;
- desenhe algumas imagens com `show_labels`: as caixas estão no lugar e com o nome certo?

In [ ]:
# 1. confira os nomes dos arquivos, o número de valores por linha e os índices das classes
# 2. conte as imagens rotuladas e as caixas de cada classe
# 3. desenhe algumas imagens com show_labels

### Exercício: treino

Organize o dataset na estrutura de pastas do YOLO, em `data/meu_detector/yolo`, sorteando cerca de 20% das imagens para a validação. Cada anotação precisa ir para o mesmo conjunto da sua imagem. Depois, escreva o `data.yaml` com `write_data_yaml`. Com poucas dezenas de imagens, não sobra o bastante para um conjunto de teste separado: no fim, testamos com imagens novas.

Em seguida, faça o ajuste fino de um YOLO26 pré-treinado para o seu dataset, salvando os resultados em `runs/meu_detector`. Use o mesmo `project=runs_dir` do exemplo, para os pesos irem para essa pasta. Algumas escolhas:

- o tamanho do modelo: o `yolo26n.pt` é o mais rápido; o `yolo26s.pt` costuma ser mais preciso e ainda treina rápido em uma GPU;
- `imgsz`: o padrão de 640 é um bom começo. Objetos pequenos na foto pedem um valor maior; se faltar memória, reduza o `batch`;
- `epochs`: com poucas imagens, cada época é rápida; algo entre 30 e 100 épocas é comum;
- `freeze`: o número de camadas iniciais congeladas. Com `freeze=10`, só a parte final da rede é treinada, o que ajuda com poucas imagens.

Os resultados de cada época ficam em `runs/meu_detector/results.csv`, e as curvas, em `runs/meu_detector/results.png`.

In [ ]:
# 1. sorteie as imagens de validação (fixe a semente para a divisão ser sempre a mesma)
# 2. copie imagens e anotações para images/train, images/val, labels/train e labels/val
# 3. escreva o data.yaml com write_data_yaml e mostre o conteúdo
# 4. faça o ajuste fino:
# my_model = YOLO(...)
# my_model.train(data=..., project=runs_dir, name="meu_detector", exist_ok=True)

Nas curvas, as perdas de treino e de validação aparecem separadas em três partes, como no detector que fizemos do zero: `box_loss` (a sobreposição entre a caixa prevista e a verdadeira, medida pelo IoU), `cls_loss` (a classe) e `l1_loss` (a distância entre as bordas das duas caixas). Nas versões anteriores, como o YOLO11, a terceira perda é a `dfl_loss`, que prevê cada borda como uma distribuição sobre posições possíveis; o YOLO26 trocou essa distribuição por uma regressão direta. Se as perdas de validação sobem enquanto as de treino descem, o modelo está sobreajustando: menos épocas, mais imagens ou `freeze` maior ajudam.

### Exercício: avaliação

Avalie o seu modelo:

- mostre as curvas de treino, em `runs/meu_detector/results.png`;
- carregue os pesos da melhor época e calcule o mAP50, o mAP50-95 e a AP de cada classe no conjunto de validação;
- mostre as detecções em algumas imagens de validação e compare com as caixas que você desenhou.

In [ ]:
# 1. mostre as curvas de treino
# 2. carregue runs/meu_detector/weights/best.pt e avalie na validação, com a AP de cada classe
# 3. mostre as detecções em algumas imagens de validação

*Comente aqui os resultados: em quais situações o detector erra? Objetos pequenos, cortados, sobrepostos, com outra iluminação? Quais classes ele confunde?*

### Exercício: testando com imagens novas

As imagens de validação vieram das mesmas fontes que as de treino: os mesmos lugares, a mesma busca, o mesmo vídeo. Um teste mais honesto usa imagens novas, de outro jeito: outra sala, outra luz, outro fundo, ou fotos suas se o treino veio da internet. O `predict` aceita um caminho de arquivo ou uma URL.

O ajuste fino trocou a camada final: o modelo novo só conhece as suas classes e esqueceu as 80 do COCO. Teste também com uma foto com pessoas ou carros e veja o que ele detecta.

In [ ]:
# 1. reúna imagens novas, de outro ambiente ou de outra fonte, e mostre as detecções do seu modelo
# 2. teste com uma imagem sem nenhuma das suas classes, com pessoas ou carros

*Comente aqui: o desempenho nas imagens novas foi parecido com o da validação? O que o modelo detecta na foto sem as suas classes?*